# Clasificador de noticias del Diario Federal

Entrena un modelo chico en castellano para clasificar cada nota del Centro de Información.

**Dos tareas posibles**, se elige en la celda de configuración:

| Tarea | Qué responde | Datos de hoy |
|---|---|---|
| `alcance` | ¿La nota es nacional, de qué región, o del exterior? | 775 notas, 7 clases, etiquetas confiables |
| `tema` | ¿De qué tema habla? | 212 notas, 6 clases, **etiquetas incompletas** |

> **Antes de entrenar `tema` leé la sección "Sobre las etiquetas de tema".** El 73 % de las notas está como "General", que no es un tema: es el cajón de "no lo pude clasificar". Un modelo entrenado con eso aprende a decir "General" y parece que funciona.

## Cómo usar este notebook

1. **Cuenta gratis** en [kaggle.com](https://www.kaggle.com) (con tu cuenta de Google).
2. Creá un **Dataset** nuevo y subí `datos/entrenamiento.csv` del repo. Nombre sugerido: `diario-federal-entrenamiento`.
3. En este notebook: **Settings → Internet: On** (hace falta para bajar el modelo base) y **Accelerator: GPU T4 x2** (o la que haya).
4. **Run all**. Tarda unos 10-20 minutos con GPU.
5. Al final, descargá `modelo-final.zip` desde el panel **Output**.

Cada sesión se corta a las ~9 horas y las horas de GPU se renuevan cada semana. Para esto sobra: son minutos.

## Sobre las etiquetas de tema

El recolector clasifica por palabras clave. Lo que no matchea cae en **"General"**, que hoy son 563 de 775 notas. Eso no es una categoría periodística: es un agujero.

Entrenar así da un modelo que dice "General" a casi todo y acierta el 73 % por la razón equivocada. Por eso el notebook:

- **descarta las "General"** cuando entrenás `tema`,
- deja solo las 212 notas con tema real,
- y **compara siempre contra la línea de base** (decir siempre la clase más común). Si el modelo no le gana a eso, no aprendió nada.

Con 212 notas y 6 clases el resultado va a ser flojo. Es esperable y está bien: sirve para medir cuánto falta. La mejora real viene de etiquetar bien las 563 "General", no de tocar el modelo.

El `alcance` sí está listo: se etiqueta con evidencia dura (el medio que publica y su provincia), sin adivinar.

## 1. Instalar dependencias

Ya vienen casi todas. Esto asegura versiones compatibles.

In [ ]:
!pip install -q "transformers>=4.41" "datasets>=2.19" accelerate scikit-learn

## 2. Configuración

Lo único que hay que tocar es `TAREA`.

In [ ]:
import glob
import json
import random
import shutil

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

# ---------------------------------------------------------------- a tocar
TAREA = "alcance"        # "alcance" o "tema"
MODELO = "dccuchile/bert-base-spanish-wwm-cased"
EPOCAS = 6
TAM_LOTE = 16
TASA_APRENDIZAJE = 2e-5
MAX_LARGO = 256
# ------------------------------------------------------------------------

print("Tarea:", TAREA)
print("Modelo base:", MODELO)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  ", torch.cuda.get_device_name(0))

## 3. Cargar el dataset

Busca `entrenamiento.csv` en cualquier dataset que hayas subido.

In [ ]:
candidatos = glob.glob("/kaggle/input/**/entrenamiento.csv", recursive=True)
if not candidatos:
    raise SystemExit(
        "No encontré entrenamiento.csv.\n"
        "Subilo como Dataset a Kaggle y volvé a correr esta celda."
    )

RUTA = candidatos[0]
df = pd.read_csv(RUTA)
print("Archivo:", RUTA)
print("Filas:", len(df))
print("Columnas:", list(df.columns))
df.head(3)

## 4. Mirar los datos

Antes de entrenar, siempre mirar la distribución. Si una clase se come al resto, el modelo va a copiar eso.

In [ ]:
print("Alcance:")
print(df["alcance"].value_counts().to_string())
print()
print("Tema:")
print(df["tema"].value_counts().to_string())
print()
print("Largo del texto (caracteres):")
print(df["texto"].str.len().describe().round(0).to_string())

## 5. Preparar las clases

- Descarta las "General" si la tarea es `tema`.
- Saca las clases con menos de 8 ejemplos: no alcanzan para aprender ni para evaluar.
- Parte en entrenamiento (80 %) y prueba (20 %), cuidando que todas las clases estén en los dos lados.

In [ ]:
datos = df.dropna(subset=["texto", TAREA]).copy()
datos = datos[datos["texto"].str.len() >= 20]

if TAREA == "tema":
    antes = len(datos)
    datos = datos[datos["tema"] != "General"]
    print(f"Descartadas {antes - len(datos)} notas con tema 'General' (no es un tema, es 'no sé').")

conteo = datos[TAREA].value_counts()
pocas = conteo[conteo < 8]
if len(pocas):
    print("\nClases con menos de 8 ejemplos, quedan afuera:")
    for clase, cantidad in pocas.items():
        print(f"  {clase}: {cantidad}")
    datos = datos[datos[TAREA].isin(conteo[conteo >= 8].index)]

etiquetas = sorted(datos[TAREA].unique())
id2label = {i: etiqueta for i, etiqueta in enumerate(etiquetas)}
label2id = {etiqueta: i for i, etiqueta in id2label.items()}
datos["etiqueta"] = datos[TAREA].map(label2id)

print(f"\n{len(datos)} notas listas | {len(etiquetas)} clases")
for i, etiqueta in id2label.items():
    print(f"  {i}: {etiqueta}")

train_df, test_df = train_test_split(
    datos, test_size=0.2, stratify=datos["etiqueta"], random_state=SEMILLA
)
print(f"\nEntrenamiento: {len(train_df)} | Prueba: {len(test_df)}")

## 6. Línea de base

Antes de entrenar nada: ¿cuánto acierta el modelo más tonto posible, que dice siempre la clase más común?

Si el modelo entrenado no le gana a esto, no aprendió. Este número es el piso, no el objetivo.

In [ ]:
mas_comun = train_df["etiqueta"].mode()[0]
tontas = [mas_comun] * len(test_df)

print(f"Línea de base: decir siempre '{id2label[mas_comun]}'")
print(f"  aciertos:  {accuracy_score(test_df['etiqueta'], tontas):.3f}")
print(f"  F1 macro:  {f1_score(test_df['etiqueta'], tontas, average='macro'):.3f}")
print()
print("El F1 macro es el número importante: promedia las clases por igual.")
print("Un modelo que ignora las clases chicas puede tener buenos aciertos y mal F1 macro.")

## 7. Tokenizar

El tokenizador convierte cada texto en números que el modelo entiende. `MAX_LARGO=256` corta los textos muy largos (título + resumen suele entrar).

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained(MODELO)


def preparar(ejemplo):
    return tokenizador(ejemplo["texto"], truncation=True, max_length=MAX_LARGO)


ds_train = Dataset.from_pandas(
    train_df[["texto", "etiqueta"]].reset_index(drop=True)
).map(preparar, batched=True)

ds_test = Dataset.from_pandas(
    test_df[["texto", "etiqueta"]].reset_index(drop=True)
).map(preparar, batched=True)

print(ds_train)
print()
print("Ejemplo tokenizado:", ds_train[0]["input_ids"][:20], "...")

## 8. Modelo y pesos por clase

Arrancamos de **BETO**, un BERT entrenado en castellano, y lo ajustamos a nuestras etiquetas.

Las clases están desbalanceadas (hay regiones con 20 notas y otras con 146). La pérdida ponderada hace que el modelo no ignore las chicas.

In [ ]:
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer)

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO,
    num_labels=len(etiquetas),
    id2label=id2label,
    label2id=label2id,
)

# Peso inverso a la frecuencia: las clases con pocos ejemplos pesan más.
frecuencias = train_df["etiqueta"].value_counts().sort_index().values
pesos = len(train_df) / (len(etiquetas) * frecuencias)
pesos = torch.tensor(pesos, dtype=torch.float)

print("Peso por clase:")
for i in range(len(etiquetas)):
    print(f"  {id2label[i]:<16} x{pesos[i]:.2f}")


class EntrenadorConPesos(Trainer):
    # Trainer normal, pero con perdida ponderada por clase.

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        etiquetas_reales = inputs.pop("labels")
        salidas = model(**inputs)
        perdida = nn.CrossEntropyLoss(weight=pesos.to(salidas.logits.device))(
            salidas.logits, etiquetas_reales
        )
        if return_outputs:
            return perdida, salidas
        return perdida

## 9. Entrenar

In [ ]:
from transformers import TrainingArguments

argumentos = TrainingArguments(
    output_dir="/kaggle/working/modelo",
    learning_rate=TASA_APRENDIZAJE,
    per_device_train_batch_size=TAM_LOTE,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCAS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

entrenador = EntrenadorConPesos(
    model=modelo,
    args=argumentos,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    tokenizer=tokenizador,
    data_collator=DataCollatorWithPadding(tokenizador),
)

entrenador.train()

## 10. Evaluar

Sobre las notas que el modelo **nunca vio** en el entrenamiento.

In [ ]:
predicciones = entrenador.predict(ds_test)
y_predicho = np.argmax(predicciones.predictions, axis=1)
y_real = predicciones.label_ids

aciertos = accuracy_score(y_real, y_predicho)
f1 = f1_score(y_real, y_predicho, average="macro")

print(f"Aciertos: {aciertos:.3f}   (línea de base: {accuracy_score(test_df['etiqueta'], tontas):.3f})")
print(f"F1 macro: {f1:.3f}   (línea de base: {f1_score(test_df['etiqueta'], tontas, average='macro'):.3f})")
print()
print(classification_report(
    y_real, y_predicho,
    target_names=[id2label[i] for i in range(len(etiquetas))],
    zero_division=0,
))

## 11. Matriz de confusión

Dónde se equivoca y con qué confunde cada clase. Esto es lo que hay que mirar para mejorar los datos.

In [ ]:
import matplotlib.pyplot as plt

matriz = confusion_matrix(y_real, y_predicho)

fig, eje = plt.subplots(figsize=(9, 7))
imagen = eje.imshow(matriz, cmap="Blues")
eje.set_xticks(range(len(etiquetas)))
eje.set_xticklabels(etiquetas, rotation=45, ha="right")
eje.set_yticks(range(len(etiquetas)))
eje.set_yticklabels(etiquetas)
eje.set_xlabel("Predicho")
eje.set_ylabel("Real")
eje.set_title("Matriz de confusión (filas = real, columnas = predicho)")

for i in range(len(etiquetas)):
    for j in range(len(etiquetas)):
        if matriz[i, j]:
            eje.text(j, i, matriz[i, j], ha="center", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 12. Ver los errores concretos

Un número no dice nada. Leer diez errores dice todo: si son notas raras o etiquetas mal puestas.

In [ ]:
revision = test_df.reset_index(drop=True).copy()
revision["predicho"] = [id2label[p] for p in y_predicho]
revision["real"] = revision[TAREA]
errores = revision[revision["predicho"] != revision["real"]]

print(f"Errores: {len(errores)} de {len(revision)}\n")
for _, fila in errores.head(15).iterrows():
    print(f"- REAL: {fila['real']}  |  PREDICHO: {fila['predicho']}")
    print(f"  {fila['texto'][:200]}")
    print()

## 13. Guardar el modelo

Se guarda el modelo, el tokenizador y **el orden de las etiquetas**. Sin ese orden, las predicciones no significan nada después.

In [ ]:
SALIDA = "/kaggle/working/modelo-final"

entrenador.save_model(SALIDA)
tokenizador.save_pretrained(SALIDA)

with open(f"{SALIDA}/etiquetas.json", "w", encoding="utf-8") as f:
    json.dump(
        {"tarea": TAREA, "modelo_base": MODELO, "etiquetas": etiquetas},
        f, ensure_ascii=False, indent=2,
    )

shutil.make_archive(SALIDA, "zip", SALIDA)

print("Guardado en:", SALIDA)
print("Para descargar: panel Output -> modelo-final.zip")
print()
for archivo in sorted(os.listdir(SALIDA)):
    tamano = os.path.getsize(os.path.join(SALIDA, archivo)) / 1e6
    print(f"  {archivo:<30} {tamano:6.1f} MB")

## Cómo seguir

**Si el F1 macro le gana a la línea de base**, el camino es:

1. Descargar `modelo-final.zip` (~450 MB).
2. Guardarlo en un **Release de GitHub** del repo `centro-informacion` (hasta 2 GB por archivo, gratis).
3. Un script en `scripts/` lo baja, carga con `transformers` y clasifica las notas nuevas.
4. Eso corre en **GitHub Actions** sin gastar deploys de Vercel: el modelo no se publica en el sitio, solo se usa para etiquetar los datos.

**Si no le gana**, no es culpa del modelo ni de Kaggle. Son las etiquetas. El siguiente paso es etiquetar bien las 563 notas "General" con una taxonomía periodística de verdad, y volver a correr esto.

**El modelo no reemplaza al editor.** Marca pistas y ordena el trabajo. La verificación sigue siendo humana.